# 複数画像での一括検証（Jupyter版）
`colab/crystal_dataset_pipeline.ipynb` と同じフォルダ構成（`IMAGE_ROOT`の各サブフォルダに画像、
`CSV_ROOT`の対応するサブフォルダに手作業計測CSV）を使って、複数の画像に対してYOLO+ResNet
パイプラインを一括実行し、手作業CSVとの誤差をまとめて検証する。`inference/`フォルダを
カレントディレクトリとしてこのnotebookを実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
%matplotlib inline
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import matplotlib.pyplot as plt

# matplotlibの既定フォント(DejaVu Sans)は日本語グリフを持たないため、
# グラフのラベルが□□□になるのを防ぐために日本語対応フォントを指定する。
for _font in ["Yu Gothic", "Meiryo", "MS Gothic"]:
    if _font in {f.name for f in plt.matplotlib.font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

from resnet_utils import build_model, make_resnet_transform

IMAGE_ROOT = "path/to/画像データ"   # ← 実際のフォルダパスに変更
CSV_ROOT = "path/to/Excelデータ"    # ← 実際のフォルダパスに変更
YOLO_WEIGHTS = "../yolo_project/runs/detect/crystal_yolo/weights/best.pt"
RESNET_WEIGHTS = "../vscode_project/outputs/best_model.pth"

PATCH_SIZE = 640
OVERLAP_RATIO = 0.15
CONF = 0.25
OUTPUT_CSV = "results/validation_results.csv"

COL_DIAMETER = "円相当径"
IMG_EXTS = (".bmp", ".tif", ".tiff", ".jpg", ".png")
EXCLUDE_MINUTES = {"0"}
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}
BASE_UM_PER_PIXEL = MAG_TO_UM_PER_PIXEL[40]
MIN_GT_DIAMETER_PX = 10  # crystal_dataset_pipeline.ipynb の MIN_SIZE と同じ考え方（ノイズ除外）

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")

assert Path(IMAGE_ROOT).exists(), f"{IMAGE_ROOT} が見つかりません。IMAGE_ROOTを実際のパスに変更してください。"
assert Path(CSV_ROOT).exists(), f"{CSV_ROOT} が見つかりません。CSV_ROOTを実際のパスに変更してください。"
assert Path(YOLO_WEIGHTS).exists(), f"{YOLO_WEIGHTS} が見つかりません。"
assert Path(RESNET_WEIGHTS).exists(), f"{RESNET_WEIGHTS} が見つかりません。"
print("準備完了")

## 画像・CSVの対応付け
`crystal_dataset_pipeline.ipynb`と同じロジック（フォルダ名の正規化、`◯分_◯倍`のパターンマッチ、
「0分」データの除外）。

In [ ]:
def normalize_folder_name(name):
    return re.sub(r"[（）()・\s]", "", name)

def get_minute_mag_key(filename):
    m = re.search(r"(\d+)分.*?(\d+)倍", filename)
    return m.groups() if m else None

def find_image_csv_pairs(image_root, csv_root):
    img_dirs = {normalize_folder_name(d): d
                for d in os.listdir(image_root)
                if os.path.isdir(os.path.join(image_root, d))}
    csv_dirs = {normalize_folder_name(d): d
                for d in os.listdir(csv_root)
                if os.path.isdir(os.path.join(csv_root, d))}
    common = set(img_dirs) & set(csv_dirs)
    print(f"画像フォルダ: {len(img_dirs)}件  CSVフォルダ: {len(csv_dirs)}件  対応: {len(common)}件")

    pairs = []
    for key in sorted(common):
        image_dir = os.path.join(image_root, img_dirs[key])
        csv_dir = os.path.join(csv_root, csv_dirs[key])

        images = defaultdict(list)
        for f in os.listdir(image_dir):
            if f.lower().endswith(IMG_EXTS):
                k = get_minute_mag_key(f)
                if k:
                    images[k].append(f)

        csvs = defaultdict(list)
        for f in os.listdir(csv_dir):
            if f.lower().endswith(".csv"):
                k = get_minute_mag_key(f)
                if k:
                    csvs[k].append(f)

        for k in sorted(set(images) & set(csvs)):
            minute, mag = k
            if minute in EXCLUDE_MINUTES:
                continue
            mag = int(mag)
            if mag not in MAG_TO_UM_PER_PIXEL:
                continue
            for img_f, csv_f in zip(sorted(images[k]), sorted(csvs[k])):
                pairs.append((os.path.join(image_dir, img_f), os.path.join(csv_dir, csv_f), mag))

    print(f"画像・CSVペア: {len(pairs)}件")
    return pairs

pairs = find_image_csv_pairs(IMAGE_ROOT, CSV_ROOT)

## モデル読み込み（1回だけ）

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=YOLO_WEIGHTS,
    confidence_threshold=CONF,
    device=str(device),
)

resnet = build_model().to(device)
resnet.load_state_dict(torch.load(RESNET_WEIGHTS, map_location=device))
resnet.eval()
transform = make_resnet_transform()

print("モデル読み込み完了")

## 一括検証

In [ ]:
def load_gt_diameters(csv_path, magnification):
    gt_df = pd.read_csv(csv_path, encoding="cp932")
    diameter_col = next((c for c in gt_df.columns if COL_DIAMETER in str(c)), None)
    if diameter_col is None:
        return np.array([])
    px = gt_df[diameter_col].dropna().astype(float)
    # crystal_dataset_pipeline.ipynb の MIN_SIZE と同じ考え方で、
    # 閾値処理由来のノイズ（極小サイズの誤検出）を除外する
    px = px[px >= MIN_GT_DIAMETER_PX]
    return (px * MAG_TO_UM_PER_PIXEL[magnification]).to_numpy()


def predict_diameters(image_path, magnification):
    sliced_result = get_sliced_prediction(
        str(image_path), detection_model,
        slice_height=PATCH_SIZE, slice_width=PATCH_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    orig_img = Image.open(image_path).convert("RGB")
    W, H = orig_img.size
    crop_scale = MAG_TO_UM_PER_PIXEL[magnification] / BASE_UM_PER_PIXEL

    preds = []
    with torch.no_grad():
        for pred in sliced_result.object_prediction_list:
            x1 = max(0, int(pred.bbox.minx))
            y1 = max(0, int(pred.bbox.miny))
            x2 = min(W, int(pred.bbox.maxx))
            y2 = min(H, int(pred.bbox.maxy))
            if x2 <= x1 or y2 <= y1:
                continue
            crop = orig_img.crop((x1, y1, x2, y2))
            if crop_scale != 1.0:
                new_w = max(1, round(crop.width * crop_scale))
                new_h = max(1, round(crop.height * crop_scale))
                crop = crop.resize((new_w, new_h))
            x = transform(crop).unsqueeze(0).to(device)
            preds.append(resnet(x).item())
    return np.array(preds)


rows = []
for i, (image_path, csv_path, mag) in enumerate(pairs):
    print(f"[{i + 1}/{len(pairs)}] {os.path.basename(image_path)} ({mag}倍)")
    try:
        gt = load_gt_diameters(csv_path, mag)
        pred = predict_diameters(image_path, mag)
    except Exception as e:
        print(f"  [スキップ] {e}")
        continue

    if len(gt) == 0 or len(pred) == 0:
        print(f"  [スキップ] 手作業{len(gt)}件 / 予測{len(pred)}件")
        continue

    diff_pct = (pred.mean() / gt.mean() - 1) * 100
    count_diff_pct = (len(pred) / len(gt) - 1) * 100
    rows.append({
        "image": os.path.basename(image_path),
        "magnification": mag,
        "n_manual": len(gt),
        "n_pred": len(pred),
        "mean_manual_um": gt.mean(),
        "mean_pred_um": pred.mean(),
        "mean_diff_pct": diff_pct,
        "median_manual_um": np.median(gt),
        "median_pred_um": np.median(pred),
        "count_diff_pct": count_diff_pct,
    })
    print(f"  手作業: {len(gt)}件 平均{gt.mean():.2f}µm  /  "
          f"予測: {len(pred)}件 平均{pred.mean():.2f}µm  差{diff_pct:+.1f}%")

result_df = pd.DataFrame(rows)
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n保存: {out_path}")

if len(result_df):
    print("\n=== 全体サマリー（画像単位の平均） ===")
    print(f"平均粒径の差: 平均{result_df['mean_diff_pct'].mean():+.2f}%  "
          f"標準偏差{result_df['mean_diff_pct'].std():.2f}%")
    print(f"件数の差: 平均{result_df['count_diff_pct'].mean():+.2f}%")

## 可視化

In [ ]:
if len(result_df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(result_df["image"], result_df["mean_diff_pct"])
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_ylabel("平均粒径の差 (%)")
    axes[0].set_title("画像ごとの平均粒径の差")
    axes[0].tick_params(axis="x", rotation=75)

    axes[1].scatter(result_df["mean_manual_um"], result_df["mean_pred_um"])
    lims = [0, max(result_df["mean_manual_um"].max(), result_df["mean_pred_um"].max()) * 1.1]
    axes[1].plot(lims, lims, "r--", linewidth=1)
    axes[1].set_xlabel("手作業(正解) 平均粒径 (µm)")
    axes[1].set_ylabel("YOLO+ResNet 平均粒径 (µm)")
    axes[1].set_title("画像ごとの平均粒径: 手作業 vs 自動")

    plt.tight_layout()
    plt.show()
else:
    print("結果が空のため、グラフは表示できません。")